In [1]:
import numpy as np
import pandas as pd
import torch

import sys
sys.path.append('../')
from utilities import binning_equal_q

In [6]:
def infer_fields(data, counts):
    
    fields = np.zeros([7,20])
    for pos in range(7):
        for color in range(20):
            fields[pos,color] = (counts*((data[:,pos] == color).astype('int'))).sum() 

    print(counts.sum(), fields.sum(1))

    fields = fields[:,:] / fields[:,0][:,np.newaxis]
    return torch.tensor(np.log(fields))

def sorted_log10p_vector_from_fields(fields):
    
    lengths = [torch.arange(20, dtype=torch.int8) for i in range(7)]
    all_seq = torch.cartesian_prod(*lengths)
    
    all_seq = all_seq.long()
    
    # generate p_vector
    p_vector = torch.zeros(20**7,dtype=torch.float32)

    for i in range(7):
        p_vector += fields[i,all_seq[:,i]]
        print(i)
    
    p_vector = torch.exp(p_vector)
    
    Z = torch.exp(fields).sum(1).prod()
    
    p_vector /= Z
    print(p_vector.sum())
    
    return torch.log(p_vector).sort()[0] / np.log(10)

def log10q_vector_func(data, fields):
    
    # generate q_vector
    q_vector = torch.zeros(len(data))

    for i in range(7):
        q_vector += fields[i,data[:,i]]
        print(i)

    q_vector = torch.exp(q_vector)

    Z = torch.exp(fields).sum(1).prod()

    q_vector /= Z

    print(q_vector.sum())
    return np.log10(q_vector.numpy())

In [4]:
data = pd.read_csv('/home/tommaso/AAV_new_data/Twist_T0.csv',index_col=0)
data = data.iloc[1:,:]

counts = data['T0'].to_numpy()
data = data.iloc[:,:7].to_numpy()

In [7]:
fields = infer_fields(data, counts)

240537189 [2.40537189e+08 2.40537189e+08 2.40537189e+08 2.40537189e+08
 2.40537189e+08 2.40537189e+08 2.40537189e+08]


In [8]:
sorted_log10p_vector = sorted_log10p_vector_from_fields(fields)

0
1
2
3
4
5
6
tensor(1.)


In [9]:
log10q_vector = log10q_vector_func(data, fields)
argsort = np.argsort(log10q_vector)
sorted_log10q_vector = log10q_vector[argsort][::-1]
counts = counts[argsort][::-1]

0
1
2
3
4
5
6
tensor(0.0808)


In [19]:
bins=5
df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins, writefolder=False)#'results_ByrneT0_IM', step=1)
df_bins.to_csv('df_bins_Twist_IM_%dbins.csv'%bins)

0
tensor(1279999999) tensor(1083569547)
elements in the bin: 196430452
nonzeros: 19693617
1
tensor(1083569547) tensor(854598732)
elements in the bin: 228970815
nonzeros: 19693617
2
tensor(854598732) tensor(602080380)
elements in the bin: 252518352
nonzeros: 19693617
3
tensor(602080380) tensor(324433882)
elements in the bin: 277646498
nonzeros: 19693617
4
tensor(324433882) tensor(101)
elements in the bin: 324433781
nonzeros: 19693620


## Write down F

In [17]:
df_bins = pd.read_csv('df_bins_Twist_IM_40bins.csv',index_col=0)
F_vec = np.zeros(2)
F_vec[0] = ((df_bins['var_count'] - df_bins['var_lambda']) / df_bins['mean_count']).mean()
F_vec[1] = ((df_bins['var_count'] - df_bins['var_lambda']) / df_bins['mean_count']).std()
np.savetxt('F_Twist.csv',F_vec,delimiter=',')
F_vec

array([3.93690699, 0.49965937])